# Hebrew Acronym Disambiguation: Dev Study

How do trained DictaBERT, Qwen and Gemini resolve a marked Hebrew acronym in context? We compare five arms: DictaBERT selection, and generation and selection by each LLM. All arms use the same occurrences marked `[ACR]...[/ACR]`; the LLMs share prompts and candidate orders.

Follow [local setup and run instructions](../README.md#local-setup), then set the paths and enable each system below. Run cells in order. `preview` uses invented examples; `run` executes enabled systems; `reload` opens a saved run. Use a new run ID after changing settings.

In [ ]:
from pathlib import Path
from uuid import uuid4
from IPython.display import HTML, display
from hebrew_acronyms import experimental_study as study
from hebrew_acronyms.paper_export import export_paper

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
settings = dict(
    mode="preview", run_kind="validation", run_id=str(uuid4()), root=ROOT,
    train_path=ROOT / "data/study_v1/encoder_inputs/train.csv",
    input_path=ROOT / "data/study_v1/encoder_inputs/dev.csv", expected_dev_items=62,
    validation_items=3, device="cpu", checkpoint=None, snapshot_path=None,
    enable_encoder=False, saved_encoder=None, saved_encoder_run_id=None,
    enable_qwen=False, qwen_model="qwen2.5:7b", ollama_url="http://localhost:11434",
    qwen_options={}, enable_gemini=False, gemini_model="gemini-3.8-flash",
    gemini_generation_config={"thinkingConfig": {"thinkingLevel": "low"}},
    request_timeout=120, output_root=ROOT.parent / "artifacts/study-runs",
    saved_run=None, saved_run_id=None,
    paper_source_run_id=None,  # Set the exact selected full-dev run ID to enable paper export.
)

## 1. Inputs

The qualified dev set contains 62 items. `validation` selects three and limits each enabled LLM to one request per task; `full_dev` uses all items. The saved cohort identifies the selected rows and the full dev input.

In [ ]:
rows, cohort = study.read_study_inputs(settings)
artifact = study.prepare_study(rows, cohort, settings)
run = artifact["settings"]
request_limit = 1 if run.get("run_kind") == "validation" else None
print("Run:", artifact["run_id"], "Items:", len(rows))

## 2. DictaBERT selection

Before prediction or saved-prediction reuse, compare the checkpoint's original training and development identities with current train and **full dev** inputs. The summary shows training settings, selected epoch, split sizes and match status. Full metadata remains in `checkpoint_info` and the artifact.

In [ ]:
encoder_predictions = None
if settings["mode"] == "run" and (run["enable_encoder"] or run["saved_encoder"]):
    encoder_predictions = study.predict_encoder(
        artifact, rows, checkpoint=run["checkpoint"], snapshot_path=run["snapshot_path"],
        device=run["device"], saved_path=run["saved_encoder"], saved_run_id=run["saved_encoder_run_id"],
    )
checkpoint_info, checkpoint_summary = study.checkpoint_report(artifact)
checkpoint_summary

## 3. Qwen generation

Generate an expansion from the marked sentence and target, without candidates or gold.

In [ ]:
if settings["mode"] == "run" and run["enable_qwen"]:
    qwen = study.connect_qwen(artifact)
    study.collect_responses(artifact, "generate", qwen, system="qwen",
                            limit=request_limit, on_record=study.save_study)

## 4. Qwen selection

Choose from shuffled candidates. A completed response must contain one uppercase, in-range letter after trimming. Incomplete responses retain their text and stopping reason but count as failures.

In [ ]:
if settings["mode"] == "run" and run["enable_qwen"]:
    study.collect_responses(artifact, "select", qwen, system="qwen",
                            limit=request_limit, on_record=study.save_study)

## 5. Gemini generation

Use the same generation prompt as Qwen. Only final text parts are collected; returned model versions are checked across both tasks.

In [ ]:
if settings["mode"] == "run" and run["enable_gemini"]:
    gemini = study.connect_gemini(artifact)
    study.collect_responses(artifact, "generate", gemini, system="gemini",
                            limit=request_limit, on_record=study.save_study)

## 6. Gemini selection

Use Qwen's prompt and candidate order with the same selection parser. Each provider runs independently.

In [ ]:
if settings["mode"] == "run" and run["enable_gemini"]:
    study.collect_responses(artifact, "select", gemini, system="gemini",
                            limit=request_limit, on_record=study.save_study)

## 7. Save outputs

Each attempt saves model metadata, settings, prompts, responses and failures in its run directory. This cell returns the artifact path for later reload.

In [ ]:
artifact_path = None
if settings["mode"] == "run":
    artifact_path = study.save_study(artifact)
print("Saved artifact:", artifact_path)

## 8. Results

These are preliminary **dev** results. Selection uses exact candidate–gold equality after trimming. Micro accuracy divides correct predictions by all requested items; macro averages within-type accuracies. Failures and unrun items remain in the denominator of an attempted system. An entirely unrun system is not measured; missing type metadata makes macro unavailable. Generation requires human judgment.

Expand items to inspect context, candidates, responses and failures. Disagreements compare valid selections. Change `start` to view more items; `results` contains all derived results. Training, architecture and service differences mean this comparison does not isolate the effect of model size.

In [ ]:
prediction_view, results = study.display_tables(artifact, start=0, max_items=10)
display(HTML(prediction_view))
metrics = results["metrics"]
disagreements = results["disagreements"]

## 9. Export the selected paper source

Set `paper_source_run_id` above to the exact full-dev run ID chosen for the paper. This cell then exports the current artifact, including after `reload`, without making new model calls. Preview, validation and invented fixtures cannot overwrite paper outputs. Selection numbers come from the same shared scorer as the display above; failures and unrun items remain in its denominator, partial coverage is labelled, and unjudged generation stays unavailable.

The atomic `paper/generated/current.tex` include selects one immutable bundle of PDF, table, text macros and a manifest with run, input, model and content hashes. Replacing the selected run updates these together; it does not update scientific conclusions. Build instructions verify the bundle before compilation. No export occurs until a source run is selected explicitly.


In [ ]:
paper_manifest = None
if settings["paper_source_run_id"] is not None:
    paper_manifest = export_paper(
        artifact, expected_run_id=artifact["run_id"],
        paper_source_run_id=settings["paper_source_run_id"],
        output_dir=ROOT / "paper/generated",
    )
    print("Paper source:", paper_manifest["run_id"], paper_manifest["execution_status"])


### Reproducibility

Saved artifacts support inspection without the original research files. See the [training appendix](train_dictabert.ipynb) for model fitting. Mock tests verify code behavior; local validation must establish checkpoint and service compatibility.

AI assistance contributed code, documentation and mock checks. Human annotation and scientific validation remain the researchers' responsibility.